#Step 1

In [4]:
import os
import pandas as pd

df = pd.read_csv("bbc-news-data.csv",sep="\t")

df = df.head(30)

df

,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...
5,business,006.txt,Japan narrowly escapes recession,Japan's economy teetered on the brink of a te...
6,business,007.txt,Jobs growth still slow in the US,The US created fewer jobs than expected in Ja...
7,business,008.txt,India calls for fair trade rules,"India, which attends the G7 meeting of seven ..."
8,business,009.txt,Ethiopia's crop production up 24%,Ethiopia produced 14.27 million tonnes of cro...
9,business,010.txt,Court rejects $280bn tobacco case,A US government claim accusing the country's ...


In [5]:
print("Number of articles:", len(df))
print("Columns:", df.columns.tolist())

Number of articles: 30
Columns: ['category', 'filename', 'title', 'content']


#Step 2

In [9]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate

model_name = "openai/gpt-oss-120b"

llm = init_chat_model(
    model_name,
    model_provider="groq"
)

topic_prompt = PromptTemplate(
    input_variables=["article"],
    template="""
You are a news topic classification system.

Classify the following BBC news article into exactly ONE
of these five categories:

Business
Entertainment
Politics
Sport
Tech

Examples:

Article:
"Shares in several major companies rose after investors
responded positively to the latest economic figures."

Category:
Business


Article:
"The football team secured a dramatic victory after
scoring in the final minutes of the match."

Category:
Sport


Article:
"The government announced new legislation following
a debate in parliament."

Category:
Politics


Article:
"A technology company launched a new smartphone with
an improved processor and longer battery life."

Category:
Tech


Article:
"The latest Hollywood film received widespread praise
from audiences and critics."

Category:
Entertainment


Now classify the following article.

Article:
{article}

Return ONLY the category name.
Do not provide an explanation.
"""
)


In [10]:
topic_chain = topic_prompt | llm

In [11]:
sample_article = df.iloc[0]["content"]

topic_result = topic_chain.invoke({
    "article": sample_article
})

print("Article:")
print(sample_article[:1000])

print("\n predicted Topic:")
print(topic_result.content)

Article:
 Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.  The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.  Time Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues. It hopes to increase subscribers by offering the online service free to TimeWarner internet customers and will try to sign up AOL's existing customers for

#Step 3

In [12]:
summary_prompt = PromptTemplate(
    input_variables=["article"],
    template="""
You are a professional news summarization system.

Summarize the following BBC news article in 2-3 sentences.

Requirements:
1. Capture the main points and key information.
2. Include who, what, when, where, and why when applicable.
3. Keep the summary concise and factual.
4. Do not add personal opinions or commentary.
5. Do not introduce information that is not present in the article.

Article:
{article}

Summary:
"""
)

summary_chain = summary_prompt | llm

In [13]:
summary_result = summary_chain.invoke({
    "article": sample_article
})

print("Original Article:")
print(sample_article[:1000])

print("\nSummary:")
print(summary_result.content)

Original Article:
 Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.  The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.  Time Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues. It hopes to increase subscribers by offering the online service free to TimeWarner internet customers and will try to sign up AOL's existing cust

#Step4

In [14]:
entity_prompt = PromptTemplate(
    input_variables=["article"],
    template="""
You are an information extraction system.

Extract the important entities mentioned in the following
BBC news article.

Identify entities in these three categories:

1. People
2. Organizations
3. Locations

Rules:
- Only include entities that are actually mentioned in the article.
- Do not invent entities.
- If no entity exists for a category, return an empty list.
- Return ONLY valid JSON.
- Do not include markdown or explanations.

Required JSON format:

{{
    "People": [],
    "Organizations": [],
    "Locations": []
}}

Article:
{article}
"""
)

entity_chain = entity_prompt | llm

In [15]:
entity_result = entity_chain.invoke({
    "article": sample_article
})

print("Extracted Entities:")
print(entity_result.content)

Extracted Entities:
{
    "People": ["Richard Parsons"],
    "Organizations": ["TimeWarner", "Google", "Warner Bros", "AOL", "US Securities Exchange Commission", "SEC", "Bertelsmann", "AOL Europe"],
    "Locations": ["US"]
}


#Step5

In [17]:
import time

detected_topics = []
summaries = []
key_entities = []

BATCH_SIZE = 5
BATCH_DELAY = 10

for batch_start in range(0, len(df), BATCH_SIZE):

    batch_end = min(batch_start + BATCH_SIZE, len(df))

    batch = df.iloc[batch_start:batch_end]

    print(
        f"\nProcessing articles "
        f"{batch_start + 1}-{batch_end} of {len(df)}..."
    )

    for index, row in batch.iterrows():

        article = row["content"]

        print(f"  Processing article {index + 1}...")

        # ----------------------------------------
        # Topic Classification
        # ----------------------------------------

        topic_result = topic_chain.invoke({
            "article": article
        })

        detected_topic = topic_result.content.strip()

        # ----------------------------------------
        # Summarization
        # ----------------------------------------

        summary_result = summary_chain.invoke({
            "article": article
        })

        summary = summary_result.content.strip()

        # ----------------------------------------
        # Entity Extraction
        # ----------------------------------------

        entity_result = entity_chain.invoke({
            "article": article
        })

        entities = entity_result.content.strip()

        # ----------------------------------------
        # Store results
        # ----------------------------------------

        detected_topics.append(detected_topic)
        summaries.append(summary)
        key_entities.append(entities)

    print(f"Batch {batch_start // BATCH_SIZE + 1} completed.")

    # Wait before starting next batch
    if batch_end < len(df):
        print(f"Waiting {BATCH_DELAY} seconds before next batch...")
        time.sleep(BATCH_DELAY)


print("\nProcessing completed!")


Processing articles 1-5 of 30...
  Processing article 1...
  Processing article 2...
  Processing article 3...
  Processing article 4...
  Processing article 5...
Batch 1 completed.
Waiting 10 seconds before next batch...

Processing articles 6-10 of 30...
  Processing article 6...
  Processing article 7...
  Processing article 8...
  Processing article 9...
  Processing article 10...
Batch 2 completed.
Waiting 10 seconds before next batch...

Processing articles 11-15 of 30...
  Processing article 11...
  Processing article 12...
  Processing article 13...
  Processing article 14...
  Processing article 15...
Batch 3 completed.
Waiting 10 seconds before next batch...

Processing articles 16-20 of 30...
  Processing article 16...
  Processing article 17...
  Processing article 18...
  Processing article 19...
  Processing article 20...
Batch 4 completed.
Waiting 10 seconds before next batch...

Processing articles 21-25 of 30...
  Processing article 21...
  Processing article 22...
  

In [18]:
df["Detected_Topic"] = detected_topics
df["Summary"] = summaries
df["Key_Entities"] = key_entities

df

,category,filename,title,content,Detected_Topic,Summary,Key_Entities
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...,Business,TimeWarner reported a 76 % jump in fourth‑quar...,"{\n ""People"": [""Richard Parsons""],\n ""Or..."
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...,Business,The dollar rose to $1.2871 per euro in late‑da...,"{\n ""People"": [""Alan Greenspan"", ""Robert Si..."
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...,Business,"The owners of Russia’s former oil giant Yukos,...","{\n ""People"": [\n ""Jamie Firestone"",..."
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...,Business,British Airways reported that pre‑tax profit f...,"{\n ""People"": [""Rod Eddington"", ""Mike Powel..."
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...,Business,Shares in UK drinks and food group Allied Dome...,"{\n ""People"": [],\n ""Organizations"": [\n..."
5,business,006.txt,Japan narrowly escapes recession,Japan's economy teetered on the brink of a te...,Business,Japan’s economy barely grew by 0.1% in the thr...,"{\n ""People"": [""Heizo Takenaka"", ""Paul Shea..."
6,business,007.txt,Jobs growth still slow in the US,The US created fewer jobs than expected in Ja...,Business,"In January, the U.S. Labor Department reported...","{\n ""People"": [""President Bush"", ""Herbert H..."
7,business,008.txt,India calls for fair trade rules,"India, which attends the G7 meeting of seven ...",Business,India’s finance minister Palaniappan Chidambar...,"{\n ""People"": [\n ""Palaniappan Chida..."
8,business,009.txt,Ethiopia's crop production up 24%,Ethiopia produced 14.27 million tonnes of cro...,Business,Ethiopia’s crop output rose to 14.27 million t...,"{\n ""People"": [""Henri Josserand""],\n ""Or..."
9,business,010.txt,Court rejects $280bn tobacco case,A US government claim accusing the country's ...,Business,A U.S. Court of Appeals for the District of Co...,"{\n ""People"": [],\n ""Organizations"": [\n..."


#Part2

#Step1

In [20]:
import pandas as pd
import json
import time

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate


# Groq LLM
model_name = "openai/gpt-oss-120b"

llm = init_chat_model(
    model_name,
    model_provider="groq"
)

print("LLM initialized successfully!")

jobs_df = pd.read_csv("job_title_des.csv")
print("Original dataset shape",jobs_df.shape)

jobs_df = jobs_df.head(25).copy()

print("Dataset used for assignment:", jobs_df.shape)

display(jobs_df.head())

LLM initialized successfully!
Original dataset shape (2277, 3)
Dataset used for assignment: (25, 3)


,Unnamed: 0,Job Title,Job Description
0,0,Flutter Developer,We are looking for hire experts flutter develo...
1,1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ..."
3,3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...
4,4,Full Stack Developer,job responsibility full stack engineer – react...


In [21]:
jobs_df = jobs_df.rename(columns={
    "Job Title": "Job_Title",
    "Job Description": "Job_Description"
})

In [24]:
job_analysis_prompt = PromptTemplate(
    input_variables=[
        "job_title",
        "job_description"
    ],
    template="""
You are a professional job-posting analysis system.

Analyze the following job posting.

TASK 1:
Classify the job into exactly ONE category:

Technology/IT
Finance
Marketing
Healthcare
Education
Others

TASK 2:
Extract:

1. Required Skills/Technologies
2. Education Requirement
3. Experience Requirement

RULES:

- Only extract information explicitly mentioned.
- Do not invent information.
- If education is not mentioned, return "Not specified".
- If experience is not mentioned, return "Not specified".
- Skills must be a list.
- Return ONLY valid JSON.
- Do not use markdown.
- Do not provide explanations.

Return exactly:

{{
    "Predicted_Category": "Technology/IT",
    "Required_Skills": [],
    "Education_Required": "Not specified",
    "Experience_Required": "Not specified"
}}

JOB TITLE:
{job_title}

JOB DESCRIPTION:
{job_description}
"""
)

job_analysis_chain = job_analysis_prompt | llm

In [25]:
sample_job = jobs_df.iloc[0]

sample_result = job_analysis_chain.invoke({
    "job_title": sample_job["Job_Title"],
    "job_description": sample_job["Job_Description"]
})

print("\n===== SAMPLE OUTPUT =====")
print(sample_result.content)


===== SAMPLE OUTPUT =====
{
    "Predicted_Category": "Technology/IT",
    "Required_Skills": ["Flutter"],
    "Education_Required": "Not specified",
    "Experience_Required": "1 year (Preferred)"
}


In [26]:
predicted_categories = []
required_skills = []
education_requirements = []
experience_requirements = []

BATCH_SIZE = 5
BATCH_DELAY = 10


for batch_start in range(
    0,
    len(jobs_df),
    BATCH_SIZE
):

    batch_end = min(
        batch_start + BATCH_SIZE,
        len(jobs_df)
    )

    batch = jobs_df.iloc[
        batch_start:batch_end
    ]

    print(
        f"\nProcessing jobs "
        f"{batch_start + 1}-{batch_end} "
        f"of {len(jobs_df)}..."
    )

    for index, row in batch.iterrows():

        print(
            f"  Processing job "
            f"{index + 1}..."
        )

        result = job_analysis_chain.invoke({
            "job_title": row["Job_Title"],
            "job_description": row["Job_Description"]
        })

        response = result.content.strip()

        try:

            data = json.loads(response)

            predicted_categories.append(
                data.get(
                    "Predicted_Category",
                    "Others"
                )
            )

            required_skills.append(
                data.get(
                    "Required_Skills",
                    []
                )
            )

            education_requirements.append(
                data.get(
                    "Education_Required",
                    "Not specified"
                )
            )

            experience_requirements.append(
                data.get(
                    "Experience_Required",
                    "Not specified"
                )
            )

        except json.JSONDecodeError:

            print(
                f"⚠ Could not parse JSON "
                f"for job {index + 1}"
            )

            predicted_categories.append(
                "Others"
            )

            required_skills.append([])

            education_requirements.append(
                "Not specified"
            )

            experience_requirements.append(
                "Not specified"
            )

    print(
        f"Batch "
        f"{batch_start // BATCH_SIZE + 1} "
        f"completed."
    )

    if batch_end < len(jobs_df):

        print(
            f"Waiting {BATCH_DELAY} seconds..."
        )

        time.sleep(BATCH_DELAY)


Processing jobs 1-5 of 25...
  Processing job 1...
  Processing job 2...
  Processing job 3...
  Processing job 4...
  Processing job 5...
Batch 1 completed.
Waiting 10 seconds...

Processing jobs 6-10 of 25...
  Processing job 6...
  Processing job 7...
  Processing job 8...
  Processing job 9...
  Processing job 10...
Batch 2 completed.
Waiting 10 seconds...

Processing jobs 11-15 of 25...
  Processing job 11...
  Processing job 12...
  Processing job 13...
  Processing job 14...
  Processing job 15...
Batch 3 completed.
Waiting 10 seconds...

Processing jobs 16-20 of 25...
  Processing job 16...
  Processing job 17...
  Processing job 18...
  Processing job 19...
  Processing job 20...
Batch 4 completed.
Waiting 10 seconds...

Processing jobs 21-25 of 25...
  Processing job 21...
  Processing job 22...
  Processing job 23...
  Processing job 24...
  Processing job 25...
Batch 5 completed.


In [27]:
jobs_df["Predicted_Category"] = (
    predicted_categories
)

jobs_df["Required_Skills"] = (
    required_skills
)

jobs_df["Education_Required"] = (
    education_requirements
)

jobs_df["Experience_Required"] = (
    experience_requirements
)

In [28]:
print("\n==========================================")
print("FINAL JOB ANALYSIS DATAFRAME")
print("==========================================")

display(jobs_df)


print("\n==========================================")
print("REQUIRED OUTPUT COLUMNS")
print("==========================================")

display(
    jobs_df[
        [
            "Job_Title",
            "Predicted_Category",
            "Required_Skills",
            "Education_Required",
            "Experience_Required"
        ]
    ]
)


FINAL JOB ANALYSIS DATAFRAME


,Unnamed: 0,Job_Title,Job_Description,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,0,Flutter Developer,We are looking for hire experts flutter develo...,Technology/IT,[Flutter],Not specified,1 year (Preferred)
1,1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...,Technology/IT,"[Python, Django, Flask, REST, RPC, Linux, SQL,...",Not specified,Not specified
2,2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ...",Technology/IT,"[Machine Learning, Deep Learning, Python, Java...","Graduate or M.Sc. in Computer Science, Mathema...",At least 3 years of hands-on development of co...
3,3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...,Technology/IT,"[iOS development, Objective-C, Cocoa Touch, Co...",Not specified,Published one or more iOS apps in the App Store
4,4,Full Stack Developer,job responsibility full stack engineer – react...,Technology/IT,"[React, React Native, Redux, Angular, Vue, Jav...",Not specified,"5+ years web development, 2+ years recent Reac..."
5,5,Java Developer,Software Developer - Integration*\r\nImmediate...,Technology/IT,"[C#, .NET, .NET Core, HTML5, CSS3, MsSQL, MySQ...","Bachelor's Degree in Computer Science, Informa...",2 years
6,6,Full Stack Developer,senior full stack developer \- 1800026h cwt lo...,Technology/IT,"[Node.js, Java, MongoDB, Elasticsearch, Redis,...","B.Sc degree in Computer Science, Engineering, ...",Minimum 2 years of relevant experience
7,7,JavaScript Developer,"Job Description:\r\n\r\nReactJS + NodeJs, Azur...",Technology/IT,"[ReactJS, NodeJS, Azure Functions, GraphQL, HT...","Any graduation, Any PG, Any Doctorate",3-8 years
8,8,DevOps Engineer,Main Responsibilities and Deliverables:\r\nMan...,Technology/IT,"[Bash, Ruby, Python, Java, Puppet, Chef, Cloud...",Not specified,Not specified
9,9,Software Engineer,"Overview\r\n\r\n\r\nBased in Silicon Valley, T...",Technology/IT,"[REST API, C/C++ (Linux/Unix), Python, Go, Git...","BS or MS in Computer Engineering, Computer Sci...",Minimum 7 years of software development experi...



REQUIRED OUTPUT COLUMNS


,Job_Title,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,Technology/IT,[Flutter],Not specified,1 year (Preferred)
1,Django Developer,Technology/IT,"[Python, Django, Flask, REST, RPC, Linux, SQL,...",Not specified,Not specified
2,Machine Learning,Technology/IT,"[Machine Learning, Deep Learning, Python, Java...","Graduate or M.Sc. in Computer Science, Mathema...",At least 3 years of hands-on development of co...
3,iOS Developer,Technology/IT,"[iOS development, Objective-C, Cocoa Touch, Co...",Not specified,Published one or more iOS apps in the App Store
4,Full Stack Developer,Technology/IT,"[React, React Native, Redux, Angular, Vue, Jav...",Not specified,"5+ years web development, 2+ years recent Reac..."
5,Java Developer,Technology/IT,"[C#, .NET, .NET Core, HTML5, CSS3, MsSQL, MySQ...","Bachelor's Degree in Computer Science, Informa...",2 years
6,Full Stack Developer,Technology/IT,"[Node.js, Java, MongoDB, Elasticsearch, Redis,...","B.Sc degree in Computer Science, Engineering, ...",Minimum 2 years of relevant experience
7,JavaScript Developer,Technology/IT,"[ReactJS, NodeJS, Azure Functions, GraphQL, HT...","Any graduation, Any PG, Any Doctorate",3-8 years
8,DevOps Engineer,Technology/IT,"[Bash, Ruby, Python, Java, Puppet, Chef, Cloud...",Not specified,Not specified
9,Software Engineer,Technology/IT,"[REST API, C/C++ (Linux/Unix), Python, Go, Git...","BS or MS in Computer Engineering, Computer Sci...",Minimum 7 years of software development experi...


#Bonus

In [31]:
import os
import json
import time
import pandas as pd

from dotenv import load_dotenv
from huggingface_hub import InferenceClient

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found in .env")

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

client = InferenceClient(
    api_key=HF_TOKEN,
    provider="auto"
)

print("Hugging Face client initialized.")
print("Model:", MODEL_NAME)

Hugging Face client initialized.
Model: meta-llama/Llama-3.1-8B-Instruct


In [33]:
jobs_df = pd.read_csv("job_title_des.csv")

print("Total jobs:", len(jobs_df))
print(jobs_df.columns.tolist())

Total jobs: 2277
['Unnamed: 0', 'Job Title', 'Job Description']


In [34]:
jobs_df = jobs_df.rename(columns={
    "Job Title": "Job_Title",
    "Job Description": "Job_Description"
})

print(jobs_df.columns.tolist())

['Unnamed: 0', 'Job_Title', 'Job_Description']


In [39]:
JOB_PROMPT = """
You are a professional job-posting analysis system.

Analyze the following job posting.

TASK 1:
Classify the job into exactly ONE category:

Technology/IT
Finance
Marketing
Healthcare
Education
Others

TASK 2:
Extract:

1. Required Skills/Technologies
2. Education Requirement
3. Experience Requirement

RULES:
- Only extract information explicitly mentioned in the job posting.
- Do not invent information.
- If education is not mentioned, return "Not specified".
- If experience is not mentioned, return "Not specified".
- Required skills must be a list.
- Return ONLY valid JSON.
- Do not use markdown.
- Do not provide explanations.

Return exactly:

{{
    "Predicted_Category": "Technology/IT",
    "Required_Skills": [],
    "Education_Required": "Not specified",
    "Experience_Required": "Not specified"
}}

JOB TITLE:
{job_title}

JOB DESCRIPTION:
{job_description}
"""

In [40]:
def analyze_job(job_title, job_description):

    prompt = JOB_PROMPT.format(
        job_title=str(job_title),
        job_description=str(job_description)
    )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_tokens=500
    )

    content = response.choices[0].message.content.strip()

    # Remove markdown code fences if the model adds them
    if content.startswith("```"):
        content = content.replace("```json", "")
        content = content.replace("```", "")
        content = content.strip()

    try:
        return json.loads(content)

    except json.JSONDecodeError:
        print("JSON parsing failed:")
        print(content)

        return {
            "Predicted_Category": "Others",
            "Required_Skills": [],
            "Education_Required": "Not specified",
            "Experience_Required": "Not specified"
        }

In [41]:
test_job = jobs_df.iloc[0]

test_result = analyze_job(
    test_job["Job_Title"],
    test_job["Job_Description"]
)

print(json.dumps(test_result, indent=4))

{
    "Predicted_Category": "Technology/IT",
    "Required_Skills": [
        "Flutter"
    ],
    "Education_Required": "Not specified",
    "Experience_Required": "1 year (Preferred)"
}


In [42]:
results = []

CHECKPOINT_EVERY = 25

total_jobs = len(jobs_df)

for index, row in jobs_df.iterrows():

    print(f"Processing {index + 1}/{total_jobs}")

    try:

        result = analyze_job(
            row["Job_Title"],
            row["Job_Description"]
        )

    except Exception as e:

        print(f"ERROR on job {index + 1}: {e}")

        result = {
            "Predicted_Category": "Others",
            "Required_Skills": [],
            "Education_Required": "Not specified",
            "Experience_Required": "Not specified"
        }

    results.append(result)

    # Save checkpoint
    if (index + 1) % CHECKPOINT_EVERY == 0:

        checkpoint_df = jobs_df.iloc[:index + 1].copy()

        checkpoint_df["Predicted_Category"] = [
            r["Predicted_Category"] for r in results
        ]

        checkpoint_df["Required_Skills"] = [
            r["Required_Skills"] for r in results
        ]

        checkpoint_df["Education_Required"] = [
            r["Education_Required"] for r in results
        ]

        checkpoint_df["Experience_Required"] = [
            r["Experience_Required"] for r in results
        ]

        checkpoint_df.to_csv(
            "part2_bonus_checkpoint.csv",
            index=False
        )

        print(
            f"Checkpoint saved: {index + 1}/{total_jobs}"
        )

    # Small delay
    time.sleep(0.3)

Processing 1/2277
Processing 2/2277
Processing 3/2277
Processing 4/2277
Processing 5/2277
Processing 6/2277
Processing 7/2277
Processing 8/2277
Processing 9/2277
Processing 10/2277
Processing 11/2277
Processing 12/2277
Processing 13/2277
Processing 14/2277
Processing 15/2277
ERROR on job 15: 402 Client Error: Payment Required for url: https://router.huggingface.co/novita/v3/openai/chat/completions (Request ID: Root=1-6aa7fd0b-4af5e50c3515817035bfcc77;4531b885-8e11-4b98-8c66-e7051711dfdb)

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.
Processing 16/2277
ERROR on job 16: 402 Client Error: Payment Required for url: https://router.huggingface.co/novita/v3/openai/chat/completions (Request ID: Root=1-6aa7fd0c-3e931772432194ab5afea043;8974c9bc-8388-4886-bb5a-8c9b900f0e95)

You have depleted your monthly included credits. Purchase pre-paid credits to continue usin

KeyboardInterrupt: 

In [ ]:
jobs_bonus_df = jobs_df.copy()

jobs_bonus_df["Predicted_Category"] = [
    r["Predicted_Category"] for r in results
]

jobs_bonus_df["Required_Skills"] = [
    r["Required_Skills"] for r in results
]

jobs_bonus_df["Education_Required"] = [
    r["Education_Required"] for r in results
]

jobs_bonus_df["Experience_Required"] = [
    r["Experience_Required"] for r in results
]